# Notebook 5: Transforming and Saving Data
**Time: 60-90 minutes**

### What you'll learn
- Turn raw API responses into clean rows with loops
- Load data into a pandas DataFrame — Python's go-to table
- Select columns, filter rows, sort, and derive new columns
- Export to CSV and Excel, and read your files back in
- Save raw JSON, and run the full fetch → clean → save pipeline yourself

---
## 5.1 Why transform?

When you call an API, the data comes back as **JSON**: values nested inside dictionaries and lists. That shape is ideal for machines, but it's hard to read and harder to share. Most people just want a spreadsheet — neat rows and columns they can open in Excel.

This notebook is the bridge. It's the everyday job of a data detective: take a messy API response, shape it into a clean table, and hand someone a file they can actually use. By the end you'll go from a raw `requests.get(...)` all the way to a saved CSV — and understand every step in between.

We'll build it up one piece at a time: loops to clean the data, pandas to tabulate it, and a few lines to save it.

---
## 5.2 Processing API responses with loops

Let's start with what you already know — loops. We'll fetch European countries and reshape each one into a simple, flat row.

In [ ]:
import requests

response = requests.get(
    "https://restcountries.com/v3.1/region/europe",
    params={"fields": "name,capital,population"}
)
countries = response.json()

print(f"Got {len(countries)} countries")
print(f"\nFirst item looks like: {countries[0]}")

In [ ]:
# Each country is a nested dictionary. Let's pull out just the fields we want
# into a flat row. (Remember .get() from Notebook 2 — it returns a fallback
# instead of crashing when a key is missing, handy for optional fields like capital.)
rows = []
for c in countries:
    rows.append({
        "name": c["name"]["common"],
        "capital": c["capital"][0] if c.get("capital") else "N/A",
        "population": c["population"]
    })

# Print the first 5
for row in rows[:5]:
    print(f"{row['name']}: {row['capital']} ({row['population']})")

This works, but printing rows by hand gets clumsy fast — and you can't easily sort, filter, or save it. Time to meet the tool built for exactly this.

---
## 5.3 Introducing pandas: the better way

**pandas** is Python's go-to library for working with tabular data — anything that fits in rows and columns, like a spreadsheet or a database table. It's the single most-used tool in data work, and almost every data task in Python starts by loading data into it.

Its core object is the **DataFrame**: a table where each **row** is one record and each **column** is one field. The neat part is how well it matches what we already have — a *list of dictionaries*. Hand that list to pandas and:

- each **dictionary becomes a row**
- each **key becomes a column**

So the cleaning loop we just wrote produces exactly the shape pandas wants.

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
df

One line, and the list of dictionaries becomes a clean table. Notice the column names came straight from the dictionary keys. Let's get a quick overview of what's inside.

In [ ]:
# Quick overview
print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print()
df.head(10)  # Show the first 10 rows

**Exercise:** Your turn to feel how a list of dictionaries becomes a table. Below is a small list — build a DataFrame from it so you can see the rows-and-columns relationship yourself.

```python
cities = [
    {"city": "Amsterdam", "country": "Netherlands", "population": 905234},
    {"city": "Berlin", "country": "Germany", "population": 3677472},
    {"city": "Paris", "country": "France", "population": 2102650},
]
```

Required variables:
- `mini_df` — a pandas DataFrame built from the `cities` list above (3 rows, columns `city`, `country`, `population`)

In [ ]:
# YOUR CODE HERE
# Turn the cities list into a DataFrame called mini_df.

cities = [
    {"city": "Amsterdam", "country": "Netherlands", "population": 905234},
    {"city": "Berlin", "country": "Germany", "population": 3677472},
    {"city": "Paris", "country": "France", "population": 2102650},
]

mini_df = None

mini_df

---
## 5.4 Handling nested API responses

The countries list was friendly: a plain list you could loop over directly. Real APIs are often messier, in two common ways — and each needs a different move:

1. **The list is wrapped in a key.** The response is a *dictionary*, and the rows you want live under a key like `"results"` or `"data"`. You extract that list **first**, then hand it to pandas.
2. **Each item is deeply nested.** The field you want is buried a few layers down (like `types[0]["type"]["name"]`). You dig it out **before** building the DataFrame.

Let's see both on a real API — the PokéAPI.

In [ ]:
# Pattern 1: the list is wrapped in a key.
# PokéAPI returns a dictionary — the rows live under "results".
response = requests.get(
    "https://pokeapi.co/api/v2/pokemon",
    params={"limit": 5}
)
response_data = response.json()

print("Top-level keys:", list(response_data.keys()))

# Extract the inner list FIRST, then make a DataFrame
df_pokemon = pd.DataFrame(response_data["results"])
df_pokemon

That was pattern 1 — one `["results"]` step and pandas handled the rest.

Pattern 2 is trickier: the field you want is buried inside nested lists and dictionaries. Before flattening a whole list, it helps to inspect **one** item and trace the exact path to the value you want.

In [ ]:
# Fetch one Pokémon and look at where its "type" lives
pikachu = requests.get("https://pokeapi.co/api/v2/pokemon/pikachu").json()

# The type is nested: a list -> a dict -> the "type" dict -> its "name"
print("The whole types field:", pikachu["types"])
print()
print("Digging down step by step:")
print("  types[0]                 ->", pikachu["types"][0])
print("  types[0]['type']         ->", pikachu["types"][0]["type"])
print("  types[0]['type']['name'] ->", pikachu["types"][0]["type"]["name"])

Now that we know the path — `types[0]["type"]["name"]` — we can loop over several Pokémon and pull each one into a flat row:

In [ ]:
# Build a flat row per Pokémon, digging out the nested type
names = ["pikachu", "charizard", "bulbasaur"]
rows = []
for name in names:
    p = requests.get(f"https://pokeapi.co/api/v2/pokemon/{name}").json()
    rows.append({
        "name": p["name"],
        "height": p["height"],
        "weight": p["weight"],
        "main_type": p["types"][0]["type"]["name"]  # the path we traced above
    })

df_flat = pd.DataFrame(rows)
df_flat

**Exercise:** Back to countries. Fetch the European countries again, this time also asking for each country's **area**. Loop over the response and flatten each country into a row, then build a DataFrame.

The area comes back under the key `"area"`; store it in a column called `area_km2`. Some fields can be missing, so reach for the safe `.get()` where it makes sense.

Required variables:
- `df_exercise` — a pandas DataFrame with columns `name`, `capital`, `population`, and `area_km2` (one row per European country)

In [ ]:
# YOUR CODE HERE
# 1) Fetch European countries, asking for fields: name, capital, population, area
# 2) Loop over the response, building one flat row per country
# 3) Put it all in a DataFrame called df_exercise

df_exercise = None

df_exercise

---
## 5.5 Filtering and selecting

Once your data is in a DataFrame, pandas gives you three everyday moves. We'll reuse the `df_exercise` table you just built and try each one:

- **Select columns** — keep only the columns you care about: `df[["name", "population"]]` (note the *double* brackets — a list of column names inside the `[]`).
- **Filter rows** — keep only rows that meet a condition: `df[df["population"] > 10_000_000]`. The inner part builds a column of `True`/`False` values, and the outer `df[...]` keeps only the `True` rows.
- **Sort** — reorder by a column: `df.sort_values("population", ascending=False)` puts the largest first.

Let's take them one at a time.

In [ ]:
# Reuse the DataFrame you built in 5.4 — no need to re-fetch
df = df_exercise
print(f"Working with {len(df)} countries; columns: {list(df.columns)}")
df.head()

**Select columns.** Pass a *list* of column names inside the brackets to keep just those, in that order:

In [ ]:
# Keep only the name and population columns
df_slim = df[["name", "population"]]
df_slim.head()

**Filter rows.** Write a condition on a column; pandas keeps only the rows where it is `True`:

In [ ]:
# Keep only countries with more than 10 million people
big_countries = df[df["population"] > 10_000_000]
print(f"{len(big_countries)} countries have more than 10 million people")
big_countries.head()

**Sort.** Reorder rows by a column. `ascending=False` puts the largest values on top:

In [ ]:
# Sort by population, largest first
df_sorted = df.sort_values("population", ascending=False)
df_sorted.head(10)

**Combine them.** These moves chain together — filter, then pick columns, then sort, all in one expression:

In [ ]:
# Big countries, name + population only, largest first
result = (
    df[df["population"] > 10_000_000][["name", "population"]]
    .sort_values("population", ascending=False)
)
result

**Exercise:** Time for a trickier one. **Population density** tells you how crowded a country is — people per square kilometre, i.e. `population ÷ area`. The DataFrame doesn't have that column yet, so you'll have to make it.

You can create a new column by assigning to it: `df["new_col"] = <a calculation using other columns>`. pandas runs the calculation row by row automatically.

Using `df_exercise`, add a `density` column, then find the **5 most crowded** and the **5 least crowded** European countries.

Required variables:
- `densest_5` — a DataFrame of the 5 countries with the **highest** density, sorted highest first (must include a `density` column)
- `least_dense_5` — a DataFrame of the 5 countries with the **lowest** density, sorted lowest first (must include a `density` column)

*Hint: once the `density` column exists, this is just sorting — and `.head()` / `.tail()` grab the two ends of a sorted table.*

In [ ]:
# YOUR CODE HERE
# 1) Make a copy to work in:   df = df_exercise.copy()
# 2) Add a density column (population per square km)
# 3) Sort by density to find the extremes
# 4) Take the top 5 and the bottom 5

densest_5 = None
least_dense_5 = None

print("Most crowded:")
print(densest_5)
print("\nLeast crowded:")
print(least_dense_5)

---
## 5.6 Exporting to CSV

A **CSV** (Comma-Separated Values) file is the universal data exchange format — it opens in Excel, Google Sheets, or any data tool. Saving a DataFrame to one is a single method call.

In [ ]:
df_sorted = df_exercise.sort_values("population", ascending=False)
df_sorted.to_csv("european_countries.csv", index=False)
print("Saved to european_countries.csv")

`index=False` tells pandas not to add its own row-number column.

**Where did the file go?** It lands in the folder where you launched Jupyter (your project folder). You can confirm it two ways:

- **In the Jupyter file browser** — open the tab you started from; the new file appears in the list.
- **In your IDE / file explorer** — look in the project folder (for example, VS Code's Explorer panel on the left).

Saving is only half the skill — you'll often need to read a file back in. pandas has `read_csv` for exactly that:

In [ ]:
# Read the file we just saved back into a fresh DataFrame
reloaded = pd.read_csv("european_countries.csv")
print(f"Loaded {len(reloaded)} rows back from the CSV")
reloaded.head()

---
## 5.7 Exporting to Excel

Need a real Excel file (`.xlsx`) instead? Same idea, different method:

In [ ]:
df_sorted.to_excel("european_countries.xlsx", index=False)
print("Saved to european_countries.xlsx")

---
## 5.8 Saving raw JSON

Sometimes you want to keep the *original* API response, untouched, so you can re-process it later without calling the API again:

In [ ]:
import json

with open("european_countries_raw.json", "w") as f:
    json.dump(countries, f, indent=2)

print("Saved raw API response to european_countries_raw.json")

---
## 5.9 Organizing outputs

As you save more files, drop them in a dedicated folder to keep your project tidy:

In [ ]:
import os

os.makedirs("output", exist_ok=True)  # Creates the folder if it doesn't exist yet

df_sorted.to_csv("output/european_countries.csv", index=False)
df_sorted.to_excel("output/european_countries.xlsx", index=False)
print("Saved into the output/ folder")

---
## 5.10 The complete pattern: start to finish

Here's the whole pipeline in one place — fetch, clean, tabulate, save. This is the shape almost every data-gathering script takes, and it's the same thing the sneak peek did back in Notebook 0 — except now you understand every line.

In [ ]:
import requests
import pandas as pd
import os

# 1. Call the API
response = requests.get(
    "https://restcountries.com/v3.1/region/europe",
    params={"fields": "name,capital,population,area"}
)

if not response.ok:
    print(f"Error: {response.status_code}")
else:
    countries = response.json()

    # 2. Process into clean rows
    rows = []
    for c in countries:
        rows.append({
            "name": c["name"]["common"],
            "capital": c["capital"][0] if c.get("capital") else "N/A",
            "population": c["population"],
            "area_km2": c.get("area", 0)
        })

    # 3. Create DataFrame and sort
    df = pd.DataFrame(rows)
    df = df.sort_values("population", ascending=False)

    # 4. Save
    os.makedirs("output", exist_ok=True)
    df.to_csv("output/european_countries.csv", index=False)

    # 5. Show results
    print(f"Saved {len(df)} countries to output/european_countries.csv")
    print()
    print(df.head(10).to_string(index=False))

---
## 5.11 Put it all together

The pattern above worked on Europe. Now run the *whole* pipeline yourself on a region you haven't touched — **Asia** — adding one new step: save the raw response as JSON before you tabulate it.

**Exercise:** Using the REST Countries API for the `asia` region (fields `name,population,area`):

1. Fetch the Asian countries.
2. Save the raw response to `asia_raw.json` (use `json.dump`).
3. Build a DataFrame `df_asia` with columns `name`, `population`, and `area_km2`.
4. From it, make `big_asia`: only countries with a population over 50 million, sorted by population (largest first).
5. Save `big_asia` to `output/big_asia.csv`.

Required variables:
- `df_asia` — a DataFrame with columns `name`, `population`, `area_km2` (one row per Asian country)
- `big_asia` — a DataFrame of Asian countries with population over 50,000,000, sorted by population descending

*You've done every individual step earlier in this notebook — this just chains them.*

In [ ]:
# YOUR CODE HERE
# Follow the 5 steps in the exercise. The complete example in 5.10 is a good
# model — adapt it to the asia region and add the "save raw JSON" step.

df_asia = None
big_asia = None

print(df_asia)
print(big_asia)

---
## 5.12 Checkpoint

Run this cell to verify your work across the whole notebook.

In [ ]:
# Run this cell to check your understanding.
# It checks the variables you produced in the exercises above.

import pandas as pd

required_vars = [
    "mini_df",
    "df_exercise",
    "densest_5",
    "least_dense_5",
    "df_asia",
    "big_asia",
]

missing = [v for v in required_vars if v not in globals() or globals()[v] is None]
assert not missing, f"Missing or unfinished: {missing}. Go back and complete those exercises."

# 5.3 - mini_df: built from the cities list
assert isinstance(mini_df, pd.DataFrame), f"mini_df should be a DataFrame, got {type(mini_df).__name__}"
assert len(mini_df) == 3, f"mini_df should have 3 rows, got {len(mini_df)}"
for col in ["city", "country", "population"]:
    assert col in mini_df.columns, f"mini_df is missing the '{col}' column"

# 5.4 - df_exercise: flattened European countries with 4 columns
assert isinstance(df_exercise, pd.DataFrame), f"df_exercise should be a DataFrame, got {type(df_exercise).__name__}"
assert len(df_exercise) == 53, (
    f"df_exercise should have all 53 European countries, got {len(df_exercise)}. "
    "Did your loop process every item?"
)
for col in ["name", "capital", "population", "area_km2"]:
    assert col in df_exercise.columns, f"df_exercise is missing the '{col}' column"

# 5.5 - densest_5 / least_dense_5: density column, 5 rows each, correct extremes
for vname, dfx in [("densest_5", densest_5), ("least_dense_5", least_dense_5)]:
    assert isinstance(dfx, pd.DataFrame), f"{vname} should be a DataFrame, got {type(dfx).__name__}"
    assert len(dfx) == 5, f"{vname} should have exactly 5 rows, got {len(dfx)}"
    assert "density" in dfx.columns, f"{vname} needs a 'density' column (population / area_km2)"

dens_vals = densest_5["density"].tolist()
assert dens_vals == sorted(dens_vals, reverse=True), "densest_5 should be sorted by density, highest first"
assert densest_5["density"].min() > least_dense_5["density"].max(), (
    "densest_5 and least_dense_5 overlap — are you taking the top 5 and the bottom 5?"
)
assert densest_5.iloc[0]["name"] == "Monaco", (
    f"The most crowded European country should be Monaco, got {densest_5.iloc[0]['name']}"
)

# 5.11 - df_asia / big_asia
assert isinstance(df_asia, pd.DataFrame), f"df_asia should be a DataFrame, got {type(df_asia).__name__}"
assert len(df_asia) == 50, f"df_asia should have all 50 Asian countries, got {len(df_asia)}"
for col in ["name", "population", "area_km2"]:
    assert col in df_asia.columns, f"df_asia is missing the '{col}' column"
assert isinstance(big_asia, pd.DataFrame), f"big_asia should be a DataFrame, got {type(big_asia).__name__}"
assert (big_asia["population"] > 50_000_000).all(), "every row in big_asia should have population > 50,000,000"
pops = big_asia["population"].tolist()
assert pops == sorted(pops, reverse=True), "big_asia should be sorted by population, largest first"
assert len(big_asia) == 13, f"Expected 13 Asian countries over 50 million, got {len(big_asia)}"

import os
assert os.path.exists("output/big_asia.csv"), (
    "Expected output/big_asia.csv to exist — did you save big_asia with .to_csv()?"
)

notebook5_ready_for_clue = True
print("All checks passed — you're ready for the clue.")

---
## 5.13 Data Detective Challenge — Clue 5

Put your skills together: fetch, process, sort, and pull out one specific value.

In [ ]:
# CHALLENGE: What is the 5th largest European country by population?
#
# Steps:
# 1. Fetch European countries with name and population
# 2. Build a clean list with a loop
# 3. Create a DataFrame
# 4. Sort by population (descending)
# 5. Get the 5th country's name (hint: index 4)

assert notebook5_ready_for_clue, "Run the 5.12 checkpoint first."

import requests
import pandas as pd

response = requests.get(
    "https://restcountries.com/v3.1/region/europe",
    params={"fields": "name,population"}
)
data = response.json()

rows = []
for c in data:
    rows.append({
        "name": c["name"]["common"],
        "population": c["population"]
    })

df = pd.DataFrame(rows)
df = df.sort_values("population", ascending=False).reset_index(drop=True)

clue_5 = ___  # The name of the 5th country (index 4)

print(f"Your answer: {clue_5}")
print(f"\nTop 6 for reference:")
print(df.head(6).to_string(index=False))

# --- Verification ---
valid = ["Italy"]
if clue_5 in valid:
    print(f"\n>>> Clue 5 collected!")
    print(f'Write down: clue_5 = "{clue_5}"')
else:
    print("Not quite — remember index 4 is the 5th item (counting starts at 0).")
    print("Use: df.iloc[4]['name']")

---
**Next up:** [Notebook 6 — Errors, Troubleshooting & Final Challenge](6_errors_troubleshooting_and_finale.ipynb)